This will be deleted once Matthew updates the EDA.ipynb.  I need this to start the collaborative filtering


In [14]:
# Imports and setup
import os, zipfile, urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded ✓")


Libraries loaded ✓


In [21]:
# Ensure RAW_DIR is a real directory
if os.path.exists(RAW_DIR) and not os.path.isdir(RAW_DIR):
    os.remove(RAW_DIR)  # remove the file blocking the folder

os.makedirs(RAW_DIR, exist_ok=True)

In [22]:
# If ZIP_PATH somehow became a folder, remove it
if os.path.isdir(ZIP_PATH):
    shutil.rmtree(ZIP_PATH)

In [ ]:
# Download MovieLens Dataset

import os
import urllib.request
import zipfile
import shutil

BASE_DIR = "/workspaces/movie-recommender-system"
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
ZIP_PATH = os.path.join(RAW_DIR, "ml-latest-small.zip")
URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

# Fix: ensure RAW_DIR is actually a directory
if os.path.exists(RAW_DIR) and not os.path.isdir(RAW_DIR):
    os.remove(RAW_DIR)

os.makedirs(RAW_DIR, exist_ok=True)

# Fix: ensure ZIP_PATH is not a directory
if os.path.isdir(ZIP_PATH):
    shutil.rmtree(ZIP_PATH)

print("Downloading dataset...")
urllib.request.urlretrieve(URL, ZIP_PATH)
print("Download complete ✓")

print("Extracting files...")
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(RAW_DIR)

print(f"Success! Files are in {RAW_DIR}/ml-latest-small/")

Download complete ✓
Extracting files...
Success! Files are in /workspaces/movie-recommender-system/data/raw/ml-latest-small/


In [24]:
# Load the raw files

EXTRACT_DIR = os.path.join(RAW_DIR, "ml-latest-small")
DATA_DIR = EXTRACT_DIR

ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))
movies  = pd.read_csv(os.path.join(DATA_DIR, "movies.csv"))
tags    = pd.read_csv(os.path.join(DATA_DIR, "tags.csv"))
links   = pd.read_csv(os.path.join(DATA_DIR, "links.csv"))

print("Shape of each file:")
for name, df in [("ratings", ratings), ("movies", movies), ("tags", tags), ("links", links)]:
    print(f"  {name:10s}: {df.shape[0]:,} rows × {df.shape[1]} cols")


Shape of each file:
  ratings   : 100,836 rows × 4 cols
  movies    : 9,742 rows × 3 cols
  tags      : 3,683 rows × 4 cols
  links     : 9,742 rows × 3 cols


In [25]:
# Confirm correct path

print("Files in directory:", os.listdir(EXTRACT_DIR))

Files in directory: ['ratings.csv', 'movies.csv', 'tags.csv', 'README.txt', 'links.csv']


In [26]:
# Data Quality Audit

def audit(df, name):
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    
    print(f"Shape        : {df.shape}")
    print(f"Duplicates   : {df.duplicated().sum()}")
    
    print("\nNull counts:")
    nulls = df.isnull().sum()
    null_pct = (nulls / len(df) * 100).round(2)
    print(pd.DataFrame({"Nulls": nulls, "Percent": null_pct}))
    
    print("\nDtypes:")
    print(df.dtypes.to_string())
    
    print("\nUnique counts:")
    print(df.nunique())
    
    print("\nSample rows:")
    print(df.head(3))

# Run audits
audit(ratings, "ratings")
audit(movies,  "movies")
audit(tags,    "tags")
audit(links,   "links")


  ratings
Shape        : (100836, 4)
Duplicates   : 0

Null counts:
           Nulls  Percent
userId         0      0.0
movieId        0      0.0
rating         0      0.0
timestamp      0      0.0

Dtypes:
userId         int64
movieId        int64
rating       float64
timestamp      int64

Unique counts:
userId         610
movieId       9724
rating          10
timestamp    85043
dtype: int64

Sample rows:
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224

  movies
Shape        : (9742, 3)
Duplicates   : 0

Null counts:
         Nulls  Percent
movieId      0      0.0
title        0      0.0
genres       0      0.0

Dtypes:
movieId    int64
title        str
genres       str

Unique counts:
movieId    9742
title      9737
genres      951
dtype: int64

Sample rows:
   movieId                    title  \
0        1         Toy Story (1995)   
1        2           Jumanji (1995)   
2      

In [28]:
# Cleaning

# --- ratings ---
ratings_clean = ratings.copy()
ratings_clean.drop_duplicates(inplace=True)

# Convert timestamp to datetime
ratings_clean["timestamp"] = pd.to_datetime(ratings_clean["timestamp"], unit="s")
ratings_clean.rename(columns={"timestamp": "rated_at"}, inplace=True)

# Define valid movie IDs
movie_ids = set(movies["movieId"])

# Drop orphan ratings (movies with no metadata)
ratings_clean = ratings_clean[ratings_clean["movieId"].isin(movie_ids)]

print(f"ratings rows: {len(ratings):,}  →  {len(ratings_clean):,} after cleaning")

ratings rows: 100,836  →  100,836 after cleaning


In [29]:
movies_clean = movies.copy()
movies_clean.drop_duplicates(inplace=True)

movies_clean["year"] = (
    movies_clean["title"]
    .str.extract(r"\((\d{4})\)", expand=False)
    .astype("Int64")
)

movies_clean["title_clean"] = (
    movies_clean["title"]
    .str.replace(r"\s*\(\d{4}\)", "", regex=True)
    .str.strip()
)

movies_clean["genre_list"] = movies_clean["genres"].apply(
    lambda g: g.split("|") if g != "(no genres listed)" else []
)

print(f"movies rows: {len(movies):,}  →  {len(movies_clean):,} after cleaning")
print(movies_clean.head(3))

movies rows: 9,742  →  9,742 after cleaning
   movieId                    title  \
0        1         Toy Story (1995)   
1        2           Jumanji (1995)   
2        3  Grumpier Old Men (1995)   

                                        genres  year       title_clean  \
0  Adventure|Animation|Children|Comedy|Fantasy  1995         Toy Story   
1                   Adventure|Children|Fantasy  1995           Jumanji   
2                               Comedy|Romance  1995  Grumpier Old Men   

                                          genre_list  
0  [Adventure, Animation, Children, Comedy, Fantasy]  
1                     [Adventure, Children, Fantasy]  
2                                  [Comedy, Romance]  


In [30]:
# --- tags ---
tags_clean = tags.copy()
tags_clean.drop_duplicates(inplace=True)

# Drop nulls BEFORE string operations
tags_clean.dropna(subset=["tag"], inplace=True)

# Convert timestamp
tags_clean["timestamp"] = pd.to_datetime(tags_clean["timestamp"], unit="s")
tags_clean.rename(columns={"timestamp": "tagged_at"}, inplace=True)

# Clean text safely
tags_clean["tag"] = tags_clean["tag"].astype(str).str.strip().str.lower()

print(f"tags rows: {len(tags):,}  →  {len(tags_clean):,} after cleaning")

tags rows: 3,683  →  3,683 after cleaning


In [32]:
# Feature Engineering

# ── Movie-level aggregates ──
movie_stats = (
    ratings_clean
    .groupby("movieId")
    .agg(
        avg_rating   = ("rating", "mean"),
        rating_count = ("rating", "count"),
        rating_std   = ("rating", "std"),
        first_rated  = ("rated_at", "min"),
        last_rated   = ("rated_at", "max"),
    )
    .reset_index()
)

movie_stats["avg_rating"] = movie_stats["avg_rating"].round(4)
movie_stats["rating_std"] = movie_stats["rating_std"].fillna(0).round(4)

# Weighted popularity score
vote_weight = movie_stats["rating_count"].mean()
global_mean = ratings_clean["rating"].mean()

movie_stats["popularity_score"] = (
    (movie_stats["rating_count"] / (movie_stats["rating_count"] + vote_weight)) * movie_stats["avg_rating"]
    + (vote_weight / (movie_stats["rating_count"] + vote_weight)) * global_mean
).round(4)

print("Movie stats sample:")
print(movie_stats.sort_values("popularity_score", ascending=False).head(10))

Movie stats sample:
      movieId  avg_rating  rating_count  rating_std         first_rated  \
277       318      4.4290           317      0.7130 1996-04-17 17:08:18   
659       858      4.2891           192      0.9043 1997-01-28 17:42:29   
2224     2959      4.2729           218      0.8614 1999-12-11 19:19:22   
921      1221      4.2597           129      0.8031 1997-01-28 17:43:16   
224       260      4.2311           251      0.8720 1996-08-17 18:23:58   
46         50      4.2377           204      0.8009 1996-04-17 17:08:18   
602       750      4.2680            97      0.8071 1997-06-23 17:30:48   
913      1213      4.2500           126      0.6834 1997-03-19 11:54:21   
461       527      4.2250           220      0.9760 1996-05-23 08:33:11   
6693    58559      4.2383           149      0.7250 2008-07-21 02:54:18   

              last_rated  popularity_score  
277  2018-09-17 04:11:50            4.3996  
659  2018-09-17 04:11:53            4.2487  
2224 2018-09-17 04:

In [33]:
from sklearn.preprocessing import MultiLabelBinarizer

# Ensure clean list format
movies_clean["genre_list"] = movies_clean["genre_list"].apply(
    lambda x: x if isinstance(x, list) else []
)

mlb = MultiLabelBinarizer()

genre_matrix = pd.DataFrame(
    mlb.fit_transform(movies_clean["genre_list"]),
    columns=mlb.classes_,
    index=movies_clean.index
)

# Add movieId for joins
genre_matrix.insert(0, "movieId", movies_clean["movieId"].values)

print(f"Genre columns ({len(mlb.classes_)}): {list(mlb.classes_)}")
print(genre_matrix.head(3))

Genre columns (19): ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
   movieId  Action  Adventure  Animation  Children  Comedy  Crime  \
0        1       0          1          1         1       1      0   
1        2       0          1          0         1       0      0   
2        3       0          0          0         0       1      0   

   Documentary  Drama  Fantasy  Film-Noir  Horror  IMAX  Musical  Mystery  \
0            0      0        1          0       0     0        0        0   
1            0      0        1          0       0     0        0        0   
2            0      0        0          0       0     0        0        0   

   Romance  Sci-Fi  Thriller  War  Western  
0        0       0         0    0        0  
1        0       0         0    0        0  
2        1       0         0    0        0  


In [34]:
# ── Recency feature ──
ratings_clean["rated_at"] = pd.to_datetime(ratings_clean["rated_at"])

latest_date = ratings_clean["rated_at"].max()
ratings_clean["days_since_rated"] = (latest_date - ratings_clean["rated_at"]).dt.days

# User-level stats
user_stats = (
    ratings_clean
    .groupby("userId")
    .agg(
        user_rating_count=("rating", "count"),
        user_avg_rating=("rating", "mean"),
        user_rating_std=("rating", "std"),
    )
    .reset_index()
)

user_stats["user_avg_rating"] = user_stats["user_avg_rating"].round(4)
user_stats["user_rating_std"] = user_stats["user_rating_std"].fillna(0).round(4)

print("User stats sample:")
print(user_stats.head(5))

User stats sample:
   userId  user_rating_count  user_avg_rating  user_rating_std
0       1                232           4.3664           0.8000
1       2                 29           3.9483           0.8056
2       3                 39           2.4359           2.0906
3       4                216           3.5556           1.3142
4       5                 44           3.6364           0.9904


In [35]:
# User-Item Interaction Matrix

# ── User-Item Interaction Matrix ──
user_item_matrix = ratings_clean.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

n_users, n_movies = user_item_matrix.shape
n_ratings = user_item_matrix.count().sum()   # counts only actual ratings
sparsity = 1 - (n_ratings / (n_users * n_movies))

print(f"Matrix shape  : {n_users} users × {n_movies} movies")
print(f"Total ratings : {n_ratings:,}")
print(f"Sparsity      : {sparsity:.4%}")
print()
print("This reflects the cold-start problem: new users or new movies have little or no interaction history.")

Matrix shape  : 610 users × 9724 movies
Total ratings : 100,836
Sparsity      : 98.3000%

This reflects the cold-start problem: new users or new movies have little or no interaction history.


In [36]:
# ── Sparse version (CSR) ──
user_ids = user_item_matrix.index.to_numpy()
movie_ids = user_item_matrix.columns.to_numpy()

user_item_sparse = csr_matrix(user_item_matrix.fillna(0).to_numpy())

print(f"CSR matrix: {user_item_sparse.shape}, {user_item_sparse.nnz:,} stored values")

density = user_item_sparse.nnz / (user_item_sparse.shape[0] * user_item_sparse.shape[1])
print(f"Density: {density:.4%}")

CSR matrix: (610, 9724), 100,836 stored values
Density: 1.7000%


In [37]:
# Train/Validation/Test Split

train_list, val_list, test_list = [], [], []

for user, df in ratings_clean.sort_values("rated_at").groupby("userId"):
    n = len(df)
    train_end = int(0.8 * n)
    val_end = int(0.9 * n)
    
    train_list.append(df.iloc[:train_end])
    val_list.append(df.iloc[train_end:val_end])
    test_list.append(df.iloc[val_end:])

train = pd.concat(train_list)
val   = pd.concat(val_list)
test  = pd.concat(test_list)


In [39]:
# Save Clean Artifacts

# Save Clean Artifacts

import os

PROCESSED = os.path.join(BASE_DIR, "data", "processed")

# If a file exists where the folder should be, remove it first
if os.path.exists(PROCESSED) and not os.path.isdir(PROCESSED):
    os.remove(PROCESSED)

# Create processed folder safely
os.makedirs(PROCESSED, exist_ok=True)

# Core interaction files
train.to_csv(os.path.join(PROCESSED, "train.csv"), index=False)
val.to_csv(os.path.join(PROCESSED, "val.csv"), index=False)
test.to_csv(os.path.join(PROCESSED, "test.csv"), index=False)
ratings_clean.to_csv(os.path.join(PROCESSED, "ratings_clean.csv"), index=False)

# Movie metadata + features
movies_clean.to_csv(os.path.join(PROCESSED, "movies_clean.csv"), index=False)
movie_stats.to_csv(os.path.join(PROCESSED, "movie_stats.csv"), index=False)

# Reset index before saving if you want a clean flat file
genre_matrix.reset_index(drop=True).to_csv(
    os.path.join(PROCESSED, "genre_encoded.csv"),
    index=False
)

# User features
user_stats.to_csv(os.path.join(PROCESSED, "user_stats.csv"), index=False)

# Full user-item matrix (keep userId as index label for easier reload later)
user_item_matrix.to_csv(
    os.path.join(PROCESSED, "user_item_matrix.csv"),
    index_label="userId"
)

print(f"Saved files to: {PROCESSED}")
for f in sorted(os.listdir(PROCESSED)):
    full_path = os.path.join(PROCESSED, f)
    size_kb = os.path.getsize(full_path) / 1024
    print(f"  {f:<35} {size_kb:8.1f} KB")

Saved files to: /workspaces/movie-recommender-system/data/processed
  genre_encoded.csv                      414.3 KB
  movie_stats.csv                        622.2 KB
  movies_clean.csv                       962.1 KB
  ratings_clean.csv                     3705.6 KB
  test.csv                               382.4 KB
  train.csv                             2952.2 KB
  user_item_matrix.csv                  6142.8 KB
  user_stats.csv                          12.4 KB
  val.csv                                371.1 KB
